# Delta Lake dev-container smoke test

Run these cells from VSCode connected to the container's Jupyter kernel (`http://localhost:8888`).
Confirms: Spark starts, Delta is wired up, and the Hive metastore + Delta files persist across restarts.

**Style note:** sections 2-6 use the **PySpark DataFrame API** (`pyspark.sql.functions`, `Window`) as the
primary way to express logic. SQL is kept only where it is genuinely the better/only tool: DDL
(`CREATE DATABASE`), `MERGE INTO`, and `DESCRIBE HISTORY`. Each of those cells says why.

**This notebook is idempotent** - every cell can be re-run in any order without duplicating rows or
failing. Writes use `mode('overwrite')` rather than `append` for exactly that reason.

> If a cell fails with `Hive metastore database is not initialized` / `Required table missing : "DBS"`,
> the Postgres metastore schema was never seeded. See `server/metastore-init/README.md`.
> If a cell fails with `DELTA_CREATE_TABLE_WITH_NON_EMPTY_LOCATION`, the metastore and `../data` have
> drifted apart - do a *full* reset (`docker compose down -v && rm -rf ../data/*`), not just `down -v`.

### A log line that used to show up here (now silenced)

Every `saveAsTable` onto an existing Delta table makes Spark try a Hive-compatible schema `ALTER` first,
which Hive always rejects for a non-Hive-SerDe format like Delta, before falling back to storing the
schema in table properties - the normal, correct path for Delta. That fallback used to print a Hive stack
trace and `WARN HiveExternalCatalog: Could not alter schema of table ... in a Hive compatible way` to the
driver log on every write. It was always benign - the write fully succeeds either way - but noisy enough
to be annoying, and there's no Spark or Delta config that skips the doomed-to-fail attempt (it's
unconditional in `HiveExternalCatalog.alterTableDataSchema`). `server/log4j2.properties` now silences
just that one logger, so it no longer appears. See that file's header for the full explanation.

In [1]:
import sys

sys.path.append('/home/spark/src')  # bind-mount of ../shared

import datetime

from pyspark.sql import functions as F
from pyspark.sql.types import (
    DateType,
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
)
from pyspark.sql.window import Window
from spark_session import get_spark

spark = get_spark('delta-lake-demo')
spark.sql('SELECT current_catalog(), version()').show(truncate=False)

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c5a61601-4cad-44a9-a6d5-0a67323b1a7d;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 46ms :: artifacts dl 2ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0

+-----------------+----------------------------------------------+
|current_catalog()|version()                                     |
+-----------------+----------------------------------------------+
|spark_catalog    |3.5.3 32232e9ed33bb16b93ad58cfde8b82e0f07c0970|
+-----------------+----------------------------------------------+



## 1. Create a schema (database) and a Delta table

SQL on purpose here: `CREATE DATABASE` / `CREATE TABLE` are DDL, and PySpark exposes no DataFrame-API
equivalent for them (`spark.catalog` can list and check, but not create a database). `IF NOT EXISTS`
on both is what makes this cell safe to re-run.

In [2]:
spark.sql('CREATE DATABASE IF NOT EXISTS sandbox')
spark.sql('USE sandbox')

spark.sql('''
CREATE TABLE IF NOT EXISTS sandbox.orders (
    order_id   BIGINT,
    customer   STRING,
    amount     DOUBLE,
    order_date DATE
)
USING DELTA
''')

spark.sql('SHOW TABLES IN sandbox').show()
print('registered in metastore:', spark.catalog.tableExists('sandbox.orders'))

26/08/16 06:34:43 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/08/16 06:34:43 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/08/16 06:34:43 WARN ObjectStore: Failed to get database default, returning NoSuchObjectException
26/08/16 06:34:43 WARN ObjectStore: Failed to get database sandbox, returning NoSuchObjectException
26/08/16 06:34:43 WARN ObjectStore: Failed to get database sandbox, returning NoSuchObjectException
26/08/16 06:34:43 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
26/08/16 06:34:44 WARN ObjectStore: Failed to get database sandbox, returning NoSuchObjectException


+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|  sandbox|   orders|      false|
+---------+---------+-----------+

registered in metastore: True


26/08/16 06:34:44 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/08/16 06:34:44 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/08/16 06:34:44 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/08/16 06:34:44 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist


## 2. Write test data (DataFrame API)

Two deliberate choices:

- **Explicit `StructType`** instead of letting `createDataFrame` infer. Inference on a column whose
  first value is `None` is fragile, and the schema has to line up with the table created above.
- **`mode('overwrite')`, not `append`.** Re-running an `append` cell silently doubles your rows, which
  makes every count and assertion below meaningless on the second run.

Row 3 (`carol`) has a null `amount` and row 5 (`dave`) a negative one **on purpose** - section 3 exists
to catch them.

In [3]:
# Every field nullable=True to match the CREATE TABLE above - plain `order_id BIGINT`
# is nullable. Declaring nullable=False here instead makes saveAsTable try to push a
# narrower schema into the Hive metastore, which logs a noisy (though non-fatal)
# InvalidOperationException: 'columns have types incompatible with the existing columns'.
orders_schema = StructType([
    StructField('order_id', LongType(), nullable=True),
    StructField('customer', StringType(), nullable=True),
    StructField('amount', DoubleType(), nullable=True),
    StructField('order_date', DateType(), nullable=True),
])

rows = [
    (1, 'alice', 120.50, datetime.date(2026, 8, 1)),
    (2, 'bob',    75.00, datetime.date(2026, 8, 2)),
    (3, 'carol',   None, datetime.date(2026, 8, 2)),  # dirty: null amount
    (4, 'alice', 240.00, datetime.date(2026, 8, 2)),
    (5, 'dave',  -15.00, datetime.date(2026, 8, 3)),  # dirty: negative amount
]

orders = spark.createDataFrame(rows, schema=orders_schema)
orders.write.format('delta').mode('overwrite').saveAsTable('sandbox.orders')

orders_tbl = spark.table('sandbox.orders')
orders_tbl.printSchema()
orders_tbl.orderBy('order_id').show()

26/08/16 06:34:51 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/08/16 06:34:54 ERROR HiveAlterHandler: Failed to alter table sandbox.orders  


root
 |-- order_id: long (nullable = true)
 |-- customer: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- order_date: date (nullable = true)

+--------+--------+------+----------+
|order_id|customer|amount|order_date|
+--------+--------+------+----------+
|       1|   alice| 120.5|2026-08-01|
|       2|     bob|  75.0|2026-08-02|
|       3|   carol|  NULL|2026-08-02|
|       4|   alice| 240.0|2026-08-02|
|       5|    dave| -15.0|2026-08-03|
+--------+--------+------+----------+



## 3. Validate: split clean vs quarantine (DataFrame API)

The earlier version of this notebook did `assert bad_rows.count() == 0` on data that was seeded with a
null amount, so it raised `AssertionError` on every single run. A validation step that always fails
teaches nothing.

Instead: build one reusable boolean rule column, then **partition** the data with it. Bad rows get
quarantined for inspection rather than silently dropped, and the assertion checks the invariant that
actually matters - *the clean set is clean*, and no row was lost or duplicated by the split.

`F.coalesce(..., F.lit(False))` matters: a comparison against `NULL` yields `NULL`, not `False`, so
without it `~is_valid` would drop the null-amount row from **both** sides.

In [4]:
is_valid = F.coalesce(
    F.col('customer').isNotNull()
    & F.col('order_date').isNotNull()
    & F.col('amount').isNotNull()
    & (F.col('amount') >= 0),
    F.lit(False),
)

clean = orders_tbl.filter(is_valid)
quarantine = orders_tbl.filter(~is_valid).withColumn(
    'reject_reason',
    F.concat_ws(
        '; ',
        F.when(F.col('customer').isNull(), F.lit('customer is null')),
        F.when(F.col('order_date').isNull(), F.lit('order_date is null')),
        F.when(F.col('amount').isNull(), F.lit('amount is null')),
        F.when(F.col('amount') < 0, F.lit('amount is negative')),
    ),
)

total, n_clean, n_bad = orders_tbl.count(), clean.count(), quarantine.count()
print(f'total={total}  clean={n_clean}  quarantined={n_bad}')
quarantine.orderBy('order_id').show(truncate=False)

# Invariants that should genuinely hold:
assert n_clean + n_bad == total, 'split lost or duplicated rows'
assert clean.filter(~is_valid).count() == 0, 'clean set still contains invalid rows'
print('Validation passed: clean set is clean, and every row is accounted for.')

total=5  clean=3  quarantined=2
+--------+--------+------+----------+------------------+
|order_id|customer|amount|order_date|reject_reason     |
+--------+--------+------+----------+------------------+
|3       |carol   |NULL  |2026-08-02|amount is null    |
|5       |dave    |-15.0 |2026-08-03|amount is negative|
+--------+--------+------+----------+------------------+

Validation passed: clean set is clean, and every row is accounted for.


## 4. PySpark function sampler: row-level transformations

A tour of the most common `pyspark.sql.functions` building blocks - all column expressions, no SQL
strings. Note `select`/`withColumn` are *lazy*: nothing executes until the `show()` at the end.

In [ ]:
enriched = (
    clean
    # null handling + arithmetic
    .withColumn('amount', F.round(F.coalesce('amount', F.lit(0.0)), 2))
    .withColumn('amount_with_tax', F.round(F.col('amount') * F.lit(1.11), 2))
    # conditional bucketing
    .withColumn(
        'value_band',
        F.when(F.col('amount') >= 200, F.lit('high'))
         .when(F.col('amount') >= 100, F.lit('medium'))
         .otherwise(F.lit('low')),
    )
    # strings
    .withColumn('customer_key', F.upper(F.trim('customer')))
    .withColumn('initial', F.substring('customer', 1, 1))
    # dates
    .withColumn('order_dow', F.date_format('order_date', 'E'))
    .withColumn('days_ago', F.datediff(F.current_date(), 'order_date'))
    .withColumn('month_start', F.trunc('order_date', 'month'))
    # struct / array / literal
    .withColumn('audit', F.struct(F.lit('notebook').alias('source'), F.current_timestamp().alias('loaded_at')))
)

enriched.select(
    'order_id', 'customer_key', 'amount', 'amount_with_tax',
    'value_band', 'order_dow', 'days_ago', 'month_start',
).orderBy('order_id').show(truncate=False)

## 5. Aggregations and window functions

`groupBy(...).agg(...)` collapses rows; a `Window` keeps every row while adding
aggregate/ranking context to it. Both are pure DataFrame API.

In [ ]:
daily = (
    enriched.groupBy('order_date')
    .agg(
        F.count('*').alias('order_count'),
        F.countDistinct('customer_key').alias('distinct_customers'),
        F.round(F.sum('amount'), 2).alias('revenue'),
        F.round(F.avg('amount'), 2).alias('avg_order'),
        F.max('amount').alias('max_order'),
        F.collect_list('customer_key').alias('customers'),
    )
    .orderBy('order_date')
)
daily.show(truncate=False)

# Window: rank within each day, and compare each order to that day's total.
per_day = Window.partitionBy('order_date').orderBy(F.col('amount').desc())
per_day_all = Window.partitionBy('order_date')

ranked = (
    enriched
    .withColumn('rank_in_day', F.row_number().over(per_day))
    .withColumn('day_revenue', F.round(F.sum('amount').over(per_day_all), 2))
    .withColumn('pct_of_day', F.round(F.col('amount') / F.col('day_revenue') * 100, 1))
    .withColumn('running_total', F.round(F.sum('amount').over(per_day.rowsBetween(Window.unboundedPreceding, 0)), 2))
)

ranked.select(
    'order_date', 'order_id', 'customer_key', 'amount',
    'rank_in_day', 'day_revenue', 'pct_of_day', 'running_total',
).orderBy('order_date', 'rank_in_day').show(truncate=False)

## 6. Persist the derived tables (DataFrame API)

`saveAsTable` both writes the Delta files and registers the table in the Hive metastore, so these show
up in `SHOW TABLES` and survive a container restart. `overwriteSchema` lets the shape of a derived
table change as you edit the cells above without having to drop it by hand.

In [ ]:
(quarantine.write.format('delta').mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('sandbox.orders_quarantine'))

(daily.drop('customers').write.format('delta').mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('sandbox.orders_daily'))

spark.sql('SHOW TABLES IN sandbox').show()
spark.table('sandbox.orders_daily').orderBy('order_date').show()

## 7. Upsert with `MERGE INTO` (SQL - kept deliberately)

This one stays SQL. The DataFrame-API equivalent is `delta.tables.DeltaTable.merge()`, which works
here (in-container) but has **incomplete support over Spark Connect** on delta-spark 3.2.x - so
`client/` scripts running from your Mac cannot use it. Keeping the notebook and the client on the same
SQL `MERGE` means the pattern you learn here transfers to both.

The bridge between the two worlds: build the source as a **DataFrame**, expose it with
`createOrReplaceTempView`, then reference that view from SQL. MERGE keyed on `order_id` is naturally
idempotent - re-running changes nothing the second time.

In [ ]:
updates = spark.createDataFrame(
    [
        (2, 'bob',   99.99, datetime.date(2026, 8, 2)),   # existing id -> UPDATE
        (6, 'erin', 310.00, datetime.date(2026, 8, 4)),   # new id      -> INSERT
    ],
    schema=orders_schema,
)
updates.createOrReplaceTempView('orders_updates')

spark.sql('''
MERGE INTO sandbox.orders AS t
USING orders_updates AS s
    ON t.order_id = s.order_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
''')

spark.table('sandbox.orders').orderBy('order_id').show()

## 8. Delta history and time travel

`DESCRIBE HISTORY` is SQL-only (there is no DataFrame-API equivalent), but reading an old snapshot is
pure DataFrame API via `.option('versionAsOf', n)`. This is the proof that `sandbox.orders` is a real
Delta table with a transaction log, not a temp view: the `WRITE` from section 2 and the `MERGE` from
section 7 each appear as their own version.

In [ ]:
history = spark.sql('DESCRIBE HISTORY sandbox.orders')
history.select('version', 'timestamp', 'operation').orderBy('version').show(truncate=False)

# Time travel: read the oldest retained version and diff it against current.
oldest = history.agg(F.min('version')).collect()[0][0]
latest = history.agg(F.max('version')).collect()[0][0]

as_of_oldest = spark.read.format('delta').option('versionAsOf', oldest).table('sandbox.orders')
print(f'version {oldest}: {as_of_oldest.count()} rows | version {latest}: {spark.table("sandbox.orders").count()} rows')
as_of_oldest.orderBy('order_id').show()

## 9. Prove it persists

```bash
cd server && docker compose restart
```

Then re-run the setup cell and section 1 only. `sandbox.orders` should still be listed with its data,
showing that both halves survived: the **Delta files** in the `../data` bind mount, and the **table
registration** in the `metastore_pgdata` Postgres volume.

Those two halves are what a reset has to keep consistent. `docker compose down -v` drops the metastore
volume but leaves `../data` untouched, which orphans every table directory on disk and makes the next
`CREATE TABLE` fail with `DELTA_CREATE_TABLE_WITH_NON_EMPTY_LOCATION`. Always reset both together:

```bash
cd server && docker compose down -v && rm -rf ../data/* && docker compose up -d
```